## CSCE 676 :: Data Mining and Analysis :: Texas A&M University :: Fall 2025

# Homework 3: All the things! (130 points + 20 BONUS)**

- **Due:** November 9 (Sunday), 11:59pm
- **You may use AI assistants**, but you must document your own reasoning and learning process.

***Goals of this homework:***
1. Implement classic, interpretable clustering (K‑Means/DBSCAN).
2. Train and interpret tree‑based models on structured data (Decision Tree, Random Forest).
3. Practice probabilistic deduplication with Bloom filters and evaluate error profiles.
4. Design and communicate your own analysis — pose a question, explore, and visualize insights from the CSCE 676 Interests dataset (Open Exploration).

***Submission instructions:***

You should post your notebook to Canvas (look for the homework 3 assignment there). Please name your submission **your-uin_hw3.ipynb**, so for example, my submission would be something like **555001234_hw3.ipynb**. Your notebook should be fully executed when you submit ... so run all the cells for us so we can see the output, then submit that.

***Late Days:***

As a reminder, you begin the semester with five late days. You may use as many as you like. There is no need to alert us to how many late days you are using. Just submit and we will make note of it. Also remember that once your late days are used up, homeworks will receive a 0.

***Collaboration and AI Assistance declaration:***

If you worked with someone on this homework, please be sure to mention that. Remember to include citations to any sources you use in the homework. Also tell us what AI assistant you used and how you used it.

## (REQUIRED) Collaboration and AI Assistance Declaration

### Collaboration Declaration:

*your response goes here*

### AI Assistance Declaration:

*your response goes here*

## Overview

This HW has **three independent, required parts** plus an optional BONUS part.

- **Part 1 (Jeopardy) — Clustering Core.**  
  Step‑by‑step: cleaning → K‑Means/DBSCAN → Analyses

- **Part 2 (Mushroom) — Interpretable Supervised Learning.**  
  Step‑by‑step: Decision Tree (gini vs entropy, depth sweep) → Random Forest → interpretability.

- **Part 3 (Humor) — Probabilistic Filtering & Lightweight Prediction.**  
  Step‑by‑step: derive Bloom parameters → implement Bloom → document your process.

- **BONUS (Interests) — Open Exploration and Creative Analysis.**
  Use the CSCE 676 Interests dataset to ask your own question, apply methods of your choice, and visualize or interpret what you find.

In [11]:
# Data URLs
Jeopardy_data_URL ='https://www.kaggle.com/datasets/tunguz/200000-jeopardy-questions'
Mushroom_data_URL ='https://www.kaggle.com/datasets/uciml/mushroom-classification'
Humor_data_URL ='https://www.kaggle.com/datasets/deepcontractor/200k-short-texts-for-humor-detection'

## 0. Environment Setup & Sampling (Optional)

- You may use the full datasets. Sampling is optional (for speed).
- If you sample, briefly report what you did (n/frac, whether you stratified, any seed).
- For Humor stream simulation, you may keep original order or shuffle (your choice—just state it).


In [12]:
###### sampling code (optional)
from pathlib import Path
import pandas as pd

# Edit paths if needed
JEOPARDY_PATH = Path("./JEOPARDY_CSV.csv")
MUSHROOM_PATH = Path("./mushrooms.csv")
HUMOR_PATH    = Path("./humor.csv")   # optional

def load_csv(path, **kwargs):
    if path.exists():
        return pd.read_csv(path, **kwargs)
    print(f"Warning: {path} not found.")
    return None

jeopardy = load_csv(JEOPARDY_PATH)
mushroom = load_csv(MUSHROOM_PATH)
humor    = load_csv(HUMOR_PATH)

# ====== (Optional) Sampling ======
# Leave all values as None to use the full dataset.
SAMPLE = {
    "jeopardy": {"n": None, "frac": None, "random_state": None, "stratify_col": None},  # e.g., {"n": 20000, "random_state": 42}
    "mushroom": {"n": None, "frac": None, "random_state": None, "stratify_col": None},  # e.g., {"frac": 1.0}
    "humor":    {"n": None, "frac": None, "random_state": None, "stratify_col": None},  # e.g., {"frac": 0.7, "random_state": 1}
}

def maybe_sample(df, cfg):
    """Return sampled df if n/frac set; otherwise return df. Optional stratify by a column name."""
    if df is None:
        return None
    n, frac, rs, strat = cfg.get("n"), cfg.get("frac"), cfg.get("random_state"), cfg.get("stratify_col")
    if strat and strat in df.columns and (n or frac):
        # stratified sampling (simple & proportional when using frac)
        if frac:
            return (df.groupby(strat, group_keys=False)
                      .apply(lambda g: g.sample(frac=frac, random_state=rs))
                      .reset_index(drop=True))
        # proportional n by class frequency (rounded)
        counts = df[strat].value_counts(normalize=True) * n
        parts = []
        for k, need in counts.round().astype(int).items():
            part = df[df[strat]==k].sample(n=min(need, len(df[df[strat]==k])), random_state=rs)
            parts.append(part)
        out = pd.concat(parts).reset_index(drop=True)
        return out.sample(frac=1.0, random_state=rs).reset_index(drop=True)
    # simple sampling
    if frac: return df.sample(frac=frac, random_state=rs).reset_index(drop=True)
    if n:    return df.sample(n=min(n, len(df)), random_state=rs).reset_index(drop=True)
    return df.reset_index(drop=True)

jeopardy_sample = maybe_sample(jeopardy, SAMPLE["jeopardy"])
mushroom_sample = maybe_sample(mushroom, SAMPLE["mushroom"])
humor_sample    = maybe_sample(humor,    SAMPLE["humor"]) if (humor is not None) else None

print("Jeopardy:", None if jeopardy is None else jeopardy.shape,
      "-> sample:", None if jeopardy_sample is None else jeopardy_sample.shape)
print("Mushroom:", None if mushroom is None else mushroom.shape,
      "-> sample:", None if mushroom_sample is None else mushroom_sample.shape)
print("Humor:   ", None if humor is None else humor.shape,
      "-> sample:", None if humor_sample is None else humor_sample.shape)


Jeopardy: (216930, 7) -> sample: (216930, 7)
Mushroom: (8124, 23) -> sample: (8124, 23)
Humor:    (200000, 2) -> sample: (200000, 2)


Full datasets

---

## Part 1 — Jeopardy! Questions -> Clustering and Analysis (30pts)

Do you know? _Jeopardy!_ is a popular U.S. quiz show where contestants are given answers and must respond with the questions.
> For example: Answer: “This planet is known as the Red Planet.” Correct question: “What is Mars?”

In this dataset, each question has a Category (topic), Question (text), and Answer (label).
You will analyze patterns in these texts and explore how question wording relates to categories and answers.


### Task1 - Preprocess (10 pts) — Design Choices + Rationale

- Clean the text (e.g., lowercase, remove stopwords/punctuation, optional stemming or lemmatization). You may use any pipeline you like, including optional TF-IDF features, embeddings, or whatever you like. Many packages (e.g., sklearn) come built in with some nice text processing packages.

- Briefly explain your design choices (what steps or parameters you used) and give a short rationale (why you think these choices help).

In [13]:
### 1) Preprocess & TF‑IDF (10 pts)
# Load data
df = pd.read_csv("JEOPARDY_CSV.csv")

# Standardize column names
df.columns = [c.strip().lower().replace(" ", "_") for c in df.columns]
# print(df.head())

# Drop rows with missing questions
df = df.dropna(subset=["question"]).reset_index(drop=True)

import re
from sklearn.feature_extraction.text import TfidfVectorizer

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", "", text)
    return text

df["clean_question"] = df["question"].apply(clean_text)
df["clean_category"] = df["category"].apply(clean_text)
df["clean_answer"] = df["answer"].apply(clean_text)

print(df.head())


   show_number    air_date      round                         category value  \
0         4680  2004-12-31  Jeopardy!                          HISTORY  $200   
1         4680  2004-12-31  Jeopardy!  ESPN's TOP 10 ALL-TIME ATHLETES  $200   
2         4680  2004-12-31  Jeopardy!      EVERYBODY TALKS ABOUT IT...  $200   
3         4680  2004-12-31  Jeopardy!                 THE COMPANY LINE  $200   
4         4680  2004-12-31  Jeopardy!              EPITAPHS & TRIBUTES  $200   

                                            question      answer  \
0  For the last 8 years of his life, Galileo was ...  Copernicus   
1  No. 2: 1912 Olympian; football star at Carlisl...  Jim Thorpe   
2  The city of Yuma in this state has a record av...     Arizona   
3  In 1963, live on "The Art Linkletter Show", th...  McDonald's   
4  Signer of the Dec. of Indep., framer of the Co...  John Adams   

                                      clean_question  \
0  for the last 8 years of his life galileo was u...  

I have cleaned the question column, removing punctuation and anything that isn't a number or letter (after lowercasing)

TF-IDF

In [14]:
tfidf = TfidfVectorizer(
    max_features=5000,
    stop_words="english",
    ngram_range=(1,2)
)

X_tfidf = tfidf.fit_transform(df["clean_question"])
print("TF-IDF shape:", X_tfidf.shape)

from sklearn.cluster import KMeans

k = 25
model = KMeans(n_clusters=k, random_state=42)
labels = model.fit_predict(X_tfidf)

df["TFIDF_cluster"] = labels

for i in range(k):
    print(f"\nCluster {i}:")
    sample = df[df["TFIDF_cluster"] == i].sample(5, random_state=42)
    print(sample[["clean_category", "clean_question", "clean_answer"]])


TF-IDF shape: (216930, 5000)

Cluster 0:
               clean_category  \
48435   south american beauty   
125101           modern dance   
129666        nuclear physics   
65880       weights  measures   
27080                 pirates   

                                           clean_question      clean_answer  
48435   usual term for the agricultural level seen her...          terraces  
125101  as opposed to the leaps of ballet mary wigmans...          kneeling  
129666  a metal used in making control rods for nuclea...           cadmium  
65880   one unit of this distance used in horse racing...         a furlong  
27080   its the term pirates used for the land along t...  the spanish main  

Cluster 1:
                     clean_category  \
45079                 southern food   
91614                    december 7   
210678             american history   
122651             also a body part   
116653  mr smith goes to washington   

                                           cl

Embeddings (HuggingFace sentence-transformers/all-MiniLM-L6-v2, 384dim)

In [15]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "sentence-transformers/all-MiniLM-L6-v2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0]  # (batch_size, seq_len, hidden_size)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

import numpy as np

def embed_texts(texts, batch_size=64):
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i:i+batch_size]
            encoded = tokenizer(batch, padding=True, truncation=True, return_tensors='pt', max_length=128)
            model_out = model(**encoded)
            batch_emb = mean_pooling(model_out, encoded['attention_mask'])
            embeddings.append(batch_emb)
    return torch.cat(embeddings).numpy()

embeddings = embed_texts(df["clean_question"].tolist())
print("Embeddings shape:", embeddings.shape)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

from sklearn.cluster import KMeans

n_clusters = k
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init='auto')
labels = kmeans.fit_predict(embeddings)
df["emb_cluster"] = labels

for i in range(n_clusters):
    print(f"\nCluster {i}:")
    sample = df[df["emb_cluster"] == i].sample(5, random_state=42)
    print(sample[["clean_category", "clean_question", "clean_answer"]])
# tqdm this

KeyboardInterrupt: 

### Task2 - Clustering with **K‑Means and DBSCAN** (20 pts)

- Run K-Means and `DBSCAN`.
- Report some clustering metrics to assess the quality of clustering (you may consider the Silhouette score for each method, or perhaps `ARI` and `NMI` using the Category labels as ground truth). This part is up to you and designed so you can observe the challenge of evaluating clustering.
- Visualize the clusters (e.g., 2D with `PCA` or `t-SNE`).

Briefly discuss which method performed better and why.


In [ ]:
### 2) Clustering: K‑Means & DBSCAN (20 pts)


---

## Part 2 — Mushrooms!! -> Decision Trees Classifications (60 pts)

This dataset contains descriptions of mushrooms from the `Agaricus` and `Lepiota` families.

Each sample is labeled as `edible` or `poisonous`, based on observable traits such as `cap shape`, `color`, `odor`, and `habitat`.

All features are categorical, making it a good exercise for preprocessing, encoding, and classification.

> You will `predict` whether a mushroom is edible or poisonous using `classification` methods.

### Taks1 - Decision Tree (20 pts)
- Train a Decision Tree classifier to predict whether each mushroom is edible or poisonous.
- Compare results using both criterion=`gini` and criterion=`entropy`, and sweep over different `max_depth` values.
- Plot test `accuracy` vs. `tree depth` and briefly discuss the effect of overfitting. What do you find out?

In [ ]:
### 1) Decision Tree (20 pts)


### Task2 - Random Forest (20 pts)

- Train Random Forest classifiers with different numbers of trees — e.g., `n_estimators ∈ {50, 100, 200}`.
- Compare their accuracy to your best single Decision Tree.
- Then, plot the top-10 most important features and discuss which mushroom traits seem most influential.

In [ ]:

### 2) Random Forest (20 pts)


### Task3 - Interpretability (20 pts)

- Train a small Decision Tree with max_depth=3 for easy visualization.
- Display the tree structure and manually trace 1–2 samples through the decision path.
- Explain in words why the model makes those predictions.

In [ ]:

### 3) Interpretability (20 pts)


# Add a short written explanation under this cell about why splits make sense.


---

## Part 3 — Humor Bloom Filter & Lightweight Baseline (40 pts )
This dataset contains humorous and non-humorous text snippets. Here, we `treat the texts as an incoming data stream`, where new messages keep arriving (like posts, tweets, or comments). You will use a `Bloom Filter`, a space-efficient probabilistic structure for membership testing, to detect whether a message (or phrase) has appeared before and to study how `false positives` grow as the filter fills up.
### Task1 - Bloom Filter (20 pts)

Implement a Bloom Filter for the humor text stream.
- For this part, you should insert the first `70%` of texts as the stream to load your bloom filter.
- As you recall, the two key design parameter are m (the size of the bloom filter) and k (the number of hash functions). Try different choices of m and k and find the theoretical minimum false positive rate (recall the formula we presented in class).
- Now calculate the false positives you get when you check the membership of the remaining 30%. That is, do not insert these 30%, just check IsMember. Here you will measure the `empirical FPR` and compare it with the theoretical expectation. What do you find?



In [ ]:

### 1) Bloom Filter (20 pts)



### Task2 - Most `Efficient' Bloom Filter (20 pts)
- It turns out you can optimize m and k for a given n (the number of elements to store) and p (the false positive probability) to yield a space-efficient Bloom filter. You may need to do a little research to dig into what this optimization is.
- For this part, consider p ∈ {0.5%, 1%, 2%, 5%} and the same 70% of the data as your n. What are the space-efficient m and k for these settings?
- Discuss your findings and how to interpret them.

In [ ]:

### 2) Space-efficiency (20 pts)



---

## Part 4 — Open Exploration: CSCE 676 Interests Dataset (Bonus 20 pts)

Background:

This dataset contains the self-reported interests of students in the class. Ideally, we want to find some notion of "community" in our class, but you are welcome to ask any question you like.

Your goal is to ask your own question about this data and explore it creatively using any of the techniques you’ve learned so far. You are welcome to collect any additional info you like (e.g., tags or recent videos from a YouTube channel).

Instructions:

- Formulate one clear research question (e.g., “Do people with similar YouTube interests share similar interests in cartoons?”).

- Choose any appropriate analysis method.

- Show your process and results (figures, tables, or short summaries).

- Briefly interpret your findings — what did you discover or find interesting?




> 👉 Be creative! This part is open-ended — the goal is to demonstrate your ability to ask meaningful questions and analyze real text data.

In [ ]:
# TODO: Load and explore the interests dataset
# Example: visualize frequent keywords or clusters of similar interests